### CONEXION DDBB OLIST

In [1]:
%pip install PyMySQL
from sqlalchemy import create_engine, text
import ssl

## CONEXION BBDD MYSQL ##
DB_USER = "nuclio"
DB_PASS = "nuclioTFM6"
DB_HOST = "nuclio.mysql.database.azure.com"
DB_NAME = "olist"

# Crear engine apuntando a la base 'olist'
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:3306/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True,
    connect_args={"ssl": {"cert_reqs": ssl.CERT_NONE, "check_hostname": False}} 
)

# tablas 'olist'
with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).fetchall()
    tables = [row[0] for row in tables]   # convertir a lista simple de strings
    
    print("Tablas en la base 'olist':")
    for t in tables:
        print("-", t)



Note: you may need to restart the kernel to use updated packages.
Tablas en la base 'olist':
- dash_olist_categorias_resumen
- dash_olist_demorados
- dash_olist_sellers
- dash_olist_states
- dash_olist_ventas_meses
- dash_sentiment_analysis
- distribucion_pedidos
- olist_customers_dataset
- olist_geolocation_dataset
- olist_order_items_dataset
- olist_order_payments_dataset
- olist_order_reviews_dataset
- olist_orders_dataset
- olist_products_dataset
- olist_sellers_dataset
- pedidos_por_tiempo
- product_category_name_translation


In [2]:
import pandas as pd
from IPython.display import display, Markdown

# --- Cargar datos base incluyendo fechas de entrega ---
query = """
SELECT 
    pct.product_category_name_english AS product_category_name,
    o.order_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p
    ON i.product_id = p.product_id
LEFT JOIN product_category_name_translation pct
    ON p.product_category_name = pct.product_category_name
LEFT JOIN olist_orders_dataset o
    ON i.order_id = o.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
"""
df = pd.read_sql_query(query, con=engine)

# --- Conversión de fechas ---
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"], errors="coerce")
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"], errors="coerce")

# --- Filtrar años de interés ---
df["order_year"] = df["order_purchase_timestamp"].dt.year
df = df[df["order_year"].isin([2017, 2018])]

# --- Calcular días de retraso ---
df["delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days

# --- Clasificar entregas ---
df["is_delayed"] = df["delay_days"] > 0
df["is_ontime"] = df["delay_days"] <= 0

# --- Agrupar por categoría y año ---
pedidos_por_estado = (
    df.groupby(["order_year", "product_category_name"], dropna=False)
      .agg(
          TotalOrders=("order_id", "nunique"),
          OnTimeOrders=("is_ontime", "sum"),
          DelayedOrders=("is_delayed", "sum")
      )
      .reset_index()
)

# --- Convertir a formato largo (ideal para Looker Studio) ---
pedidos_looker = pd.melt(
    pedidos_por_estado,
    id_vars=["order_year", "product_category_name"],
    value_vars=["OnTimeOrders", "DelayedOrders"],
    var_name="delivery_status",
    value_name="OrdersQty"
)

pedidos_looker["delivery_status"] = pedidos_looker["delivery_status"].replace({
    "OnTimeOrders": "On Time",
    "DelayedOrders": "Delayed"
})

# --- Ordenar por año y categoría ---
pedidos_looker = pedidos_looker.sort_values(by=["order_year", "product_category_name"])

# --- Mostrar resultados ---
display(Markdown("### Pedidos a tiempo vs demorados — Formato optimizado para Looker Studio"))
display(pedidos_looker.head(20))


### Pedidos a tiempo vs demorados — Formato optimizado para Looker Studio

,order_year,product_category_name,delivery_status,OrdersQty
0,2017,agro_industry_and_commerce,On Time,59
143,2017,agro_industry_and_commerce,Delayed,2
1,2017,air_conditioning,On Time,122
144,2017,air_conditioning,Delayed,4
2,2017,art,On Time,32
145,2017,art,Delayed,0
3,2017,arts_and_craftmanship,On Time,2
146,2017,arts_and_craftmanship,Delayed,0
4,2017,audio,On Time,146
147,2017,audio,Delayed,19


In [3]:
from sqlalchemy import text
from sqlalchemy.types import Integer, String

# --- Guardar DataFrame en la base de datos ---
table_name = "dash_olist_demorados"

# --- Eliminar tabla si existe (para evitar conflictos por reflexión) ---
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS `{table_name}`;"))

# --- Definir tipos de columnas ---
dtype_map = {
    "order_year": Integer(),
    "product_category_name": String(100),
    "delivery_status": String(20),
    "OrdersQty": Integer(),
}

# --- Crear y cargar tabla ---
pedidos_looker.to_sql(
    name=table_name,
    con=engine,
    if_exists="fail",     # evita sobreescritura accidental
    index=False,
    dtype=dtype_map,
    method="multi"
)

print(f"Tabla '{table_name}' creada y cargada correctamente en la base de datos.")

# --- Validación opcional: mostrar número de registros cargados ---
with engine.connect() as conn:
    result = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`;"))
    total_rows = result.scalar()
    print(f"Total de registros insertados: {total_rows:,}")


Tabla 'dash_olist_demorados' creada y cargada correctamente en la base de datos.
Total de registros insertados: 286
